# Step 3: Plan Logistics

# Built entirely by Claude Code

## Methodology

Step 2 gave us the best venue in each top city. Step 3 turns that list into an actual **tour**: which city
first, which date, how much gas, and where the crew sleeps, at the lowest total cost.

### Data

| Need | File used here (toy data) | Real source in the original project |
|---|---|---|
| Gas prices | `data/gas_prices.csv` (price per gallon by state) | AAA state averages, web scraped |
| Hotel prices | `data/hotels.csv` (6 hotels per city: rating, weekday and weekend price) | Booking.com, web scraped |
| Driving distance | estimated from coordinates (straight line × 1.2) | Google Maps API |

### Steps

1. **Pick the stops:** the best venue in each of the artist's top cities (up to 8).
2. **Distances:** the miles between every pair of stops. The straight-line distance from Step 2 (haversine)
   is multiplied by 1.2, because roads are longer than a straight line.
3. **Gas cost of each drive:** miles ÷ miles-per-gallon × the average gas price of the two states.
4. **Hotel for each city:** the cheapest hotel rated at least 3.5 (out of 5).
5. **Dates:** for a given order of cities, each show goes on the venue's first free weekend date after the
   drive (the bus can drive about 450 miles a day).
6. **Hotel cost:** the crew sleeps in a hotel every night of the tour. Friday and Saturday nights use the
   weekend price. A tour that drags on costs more nights, so shorter tours are cheaper.
7. **Pick the route:** try **every** possible order (8 cities = 40,320 orders), price each one, and keep the
   cheapest. Visiting all the cities matters more than saving money, so an order that has to skip a city
   loses to one that doesn't.
8. **Compare** with the simple plan of visiting the cities in ranking order.
9. **Export** the result for the app (`app/data.js`).

### Assumptions and limits
- Everything here is toy data, so the dollar amounts are examples, not real quotes.
- Distance is a straight line × 1.2, not a real road route, and ignores tolls and traffic.
- Trying every order only works for about 8 stops (9 stops is already 362,880 orders). More stops would
  need a faster method that gives a good, but not guaranteed best, answer.
- Hotel prices only change between weekdays and weekends, not by season or by event.
- Idle nights are spent in the city of the last show.

In [1]:
import math, itertools, bisect, json, os
from datetime import date, timedelta
import numpy as np
import pandas as pd

In [2]:
# assumptions you can change (the app lets you change the money ones too)
artist_id = 'A001'
tour_start = date(2027, 3, 1)      # same tour window as Step 2
tour_end = date(2027, 6, 30)
max_stops = 8                      # how many cities to visit (8 stops = 40,320 possible orders)
crew_size = 8                      # people travelling
rooms = math.ceil(crew_size / 2)   # 2 people share a room
mpg = 8                            # tour bus miles per gallon
miles_per_day = 450                # how far the bus can drive in one day (about 8 hours)
min_rating = 3.5                   # only book hotels rated at least this (out of 5)
road_factor = 1.2                  # roads are about 20% longer than a straight line

In [3]:
## get the input data (what Step 2 saved)
venues_best = pd.read_csv('output/step2_best_venues.csv')
venues_best.head()

,artist_id,city_id,city,state,venue_id,venue_name,venue_type,locality,capacity,expected_demand,capacity_fit,venue_lat,venue_lon,distance_miles,avg_ticket_price_usd,city_index,venue_score
0,A001,C022,Portland,OR,V0200,Echo Music Hall,music hall,Portland,5000,4684.0,1.067464,45.55441,-122.73545,3.9,72,88.164797,0.777
1,A001,C003,Chicago,IL,V0039,Northside Bowl,amphitheater,Chicago,8350,12211.0,0.683810,41.87592,-87.61949,0.6,76,81.402265,0.685
2,A001,C013,Seattle,WA,V0137,Riverside Auditorium,music hall,Seattle,5100,4746.0,1.074589,47.60479,-122.32680,0.3,50,64.623257,0.823
3,A001,C011,Boston,MA,V0117,Echo Music Hall,music hall,Boston,4850,5957.0,0.814168,42.35975,-71.05896,0.0,72,64.301310,0.740
4,A001,C025,Austin,TX,V0222,Harbor Pavilion Hall,music hall,Austin,3950,3116.0,1.267651,30.26878,-97.74106,0.2,77,58.420823,0.773


In [4]:
## the weekend dates each venue still has open
open_weekend_dates = pd.read_csv('output/step2_open_weekend_dates.csv', parse_dates=['event_date'])

# for each venue: a sorted list of its free weekend dates
open_days = {v: sorted(d.date() for d in g.event_date) for v, g in open_weekend_dates.groupby('venue_id')}
open_days['V0200'][:5]   # first free dates of one example venue (Echo Music Hall, Portland)

[datetime.date(2027, 3, 13),
 datetime.date(2027, 3, 14),
 datetime.date(2027, 3, 19),
 datetime.date(2027, 3, 20),
 datetime.date(2027, 3, 21)]

In [5]:
## get the input data
gas_prices = pd.read_csv('data/gas_prices.csv').set_index('state').regular_usd_per_gal
gas_prices.head()

state
AZ    3.75
CA    4.86
CO    3.44
DC    3.56
FL    3.38
Name: regular_usd_per_gal, dtype: float64

In [6]:
## get the input data
hotels = pd.read_csv('data/hotels.csv')
hotels.head()

,hotel_id,city_id,hotel_name,rating,price_weekday_usd,price_weekend_usd
0,HC0010,C001,Summit Hotel,3.5,179,181
1,HC0011,C001,Capitol Inn,3.6,163,196
2,HC0012,C001,Lakeside Hotel,3.7,175,212
3,HC0013,C001,Maple Inn,4.0,203,232
4,HC0014,C001,Lakeside Plaza,4.7,205,228


In [7]:
# step 1: the stops = the best venue in each of the artist's top cities
def get_stops(artist_id):
    stops = venues_best[venues_best.artist_id == artist_id].sort_values('city_index', ascending=False)
    return stops.head(max_stops).reset_index(drop=True)

stops = get_stops(artist_id)
stops[['city', 'state', 'venue_name', 'capacity', 'expected_demand', 'city_index']]

,city,state,venue_name,capacity,expected_demand,city_index
0,Portland,OR,Echo Music Hall,5000,4684.0,88.164797
1,Chicago,IL,Northside Bowl,8350,12211.0,81.402265
2,Seattle,WA,Riverside Auditorium,5100,4746.0,64.623257
3,Boston,MA,Echo Music Hall,4850,5957.0,64.301310
4,Austin,TX,Harbor Pavilion Hall,3950,3116.0,58.420823
5,Minneapolis,MN,Velvet Music Hall,4100,3898.0,57.499527
6,Los Angeles,CA,Golden Amphitheater,9800,9595.0,55.983047
7,Orlando,FL,Granite Theater,2200,2881.0,52.128410


In [8]:
# step 2: how far is every stop from every other stop?
def haversine_miles(lat1, lon1, lat2, lon2):   # same function as in Create Index
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))

def road_miles_matrix(stops):
    lat, lon = stops.venue_lat.values, stops.venue_lon.values
    # a column of stops against a row of stops gives a full stops x stops table of distances
    straight_line = haversine_miles(lat[:, None], lon[:, None], lat[None, :], lon[None, :])
    return road_factor * straight_line

miles = road_miles_matrix(stops)
pd.DataFrame(miles, index=stops.city, columns=stops.city).round(0)

city,Portland,Chicago,Seattle,Boston,Austin,Minneapolis,Los Angeles,Orlando
city,,,,,,,,
Portland,0.0,2109.0,172.0,3042.0,2054.0,1711.0,994.0,3035.0
Chicago,2109.0,0.0,2080.0,1018.0,1175.0,426.0,2092.0,1183.0
Seattle,172.0,2080.0,0.0,2982.0,2123.0,1669.0,1151.0,3056.0
Boston,3042.0,1018.0,2982.0,0.0,2033.0,1345.0,3110.0,1338.0
Austin,2054.0,1175.0,2123.0,2033.0,0.0,1254.0,1472.0,1191.0
Minneapolis,1711.0,426.0,1669.0,1345.0,1254.0,0.0,1826.0,1571.0
Los Angeles,994.0,2092.0,1151.0,3110.0,1472.0,1826.0,0.0,2640.0
Orlando,3035.0,1183.0,3056.0,1338.0,1191.0,1571.0,2640.0,0.0


In [9]:
# step 3: gas price in each stop's state
def add_gas_price(stops):
    return stops.assign(gas_price=stops.state.map(gas_prices))

stops = add_gas_price(stops)
stops[['city', 'state', 'gas_price']]

,city,state,gas_price
0,Portland,OR,4.10
1,Chicago,IL,3.60
2,Seattle,WA,4.44
3,Boston,MA,3.57
4,Austin,TX,2.95
5,Minneapolis,MN,3.16
6,Los Angeles,CA,4.86
7,Orlando,FL,3.38


In [10]:
# step 3 (continued): what does the gas for each drive cost?
def fuel_cost_matrix(miles, gas_price):
    avg_price = (gas_price.values[:, None] + gas_price.values[None, :]) / 2    # average of the two states' prices
    return miles / mpg * avg_price                                              # gallons x price per gallon

fuel = fuel_cost_matrix(miles, stops.gas_price)
pd.DataFrame(fuel, index=stops.city, columns=stops.city).round(0)

city,Portland,Chicago,Seattle,Boston,Austin,Minneapolis,Los Angeles,Orlando
city,,,,,,,,
Portland,0.0,1015.0,92.0,1458.0,905.0,776.0,556.0,1419.0
Chicago,1015.0,0.0,1045.0,456.0,481.0,180.0,1106.0,516.0
Seattle,92.0,1045.0,0.0,1493.0,981.0,793.0,669.0,1494.0
Boston,1458.0,456.0,1493.0,0.0,828.0,566.0,1639.0,581.0
Austin,905.0,481.0,981.0,828.0,0.0,479.0,719.0,471.0
Minneapolis,776.0,180.0,793.0,566.0,479.0,0.0,915.0,642.0
Los Angeles,556.0,1106.0,669.0,1639.0,719.0,915.0,0.0,1360.0
Orlando,1419.0,516.0,1494.0,581.0,471.0,642.0,1360.0,0.0


In [11]:
# step 4: pick the cheapest hotel rated at least min_rating in each city
def cheapest_hotel(city_id):
    in_city = hotels[hotels.city_id == city_id]
    good = in_city[in_city.rating >= min_rating]
    if good.empty:                                                    # nothing good enough: take the best rated one
        good = in_city.nlargest(1, 'rating')
    typical_price = (5 * good.price_weekday_usd + 2 * good.price_weekend_usd) / 7   # 5 weekday + 2 weekend nights
    return good.loc[typical_price.idxmin()]

def add_cheapest_hotel(stops):
    chosen = stops.city_id.apply(cheapest_hotel)
    return stops.assign(hotel_name=chosen.hotel_name.values, hotel_rating=chosen.rating.values,
                        price_weekday=chosen.price_weekday_usd.values, price_weekend=chosen.price_weekend_usd.values)

stops = add_cheapest_hotel(stops)
stops[['city', 'hotel_name', 'hotel_rating', 'price_weekday', 'price_weekend']]

,city,hotel_name,hotel_rating,price_weekday,price_weekend
0,Portland,Summit Motel,3.6,97,94
1,Chicago,Crown Suites,4.0,127,135
2,Seattle,Grand Motel,3.8,141,139
3,Boston,Capitol Suites,4.0,147,151
4,Austin,Harbor Motel,4.0,103,108
5,Minneapolis,Union Inn,3.5,99,123
6,Los Angeles,Capitol Inn,3.7,168,164
7,Orlando,Lakeside Lodge,3.5,99,119


In [12]:
# step 5: pack everything the search needs into plain lists and tables (makes it fast)
def prepare_inputs(stops):
    miles = road_miles_matrix(stops)
    return {
        'n': len(stops),
        'venue_ids': stops.venue_id.tolist(),
        'miles': miles,
        'fuel': fuel_cost_matrix(miles, stops.gas_price),
        'weekday': stops.price_weekday.tolist(),
        'weekend': stops.price_weekend.tolist(),
    }

inp = prepare_inputs(stops)
inp['n'], inp['venue_ids']

(8, ['V0200', 'V0039', 'V0137', 'V0117', 'V0222', 'V0140', 'V0017', 'V0184'])

In [13]:
# step 6: given an ORDER of cities, work out each show date, the gas and the hotel cost
def schedule_route(order, inp):
    shows, skipped = [], []
    for k in order:
        if shows:
            prev = shows[-1]['stop']
            drive_miles = inp['miles'][prev, k]
            drive_days = max(1, math.ceil(drive_miles / miles_per_day))
            earliest = shows[-1]['show_date'] + timedelta(days=drive_days)     # can't play before the bus arrives
            leg_fuel = inp['fuel'][prev, k]
        else:
            drive_miles, leg_fuel, earliest = 0, 0, tour_start

        # first free weekend date on or after the earliest possible date
        days = open_days[inp['venue_ids'][k]]
        i = bisect.bisect_left(days, earliest)
        if i == len(days) or days[i] > tour_end:
            skipped.append(k)                    # no date left in the window: skip this city
            continue
        shows.append({'stop': k, 'show_date': days[i], 'drive_miles': drive_miles, 'fuel_cost': leg_fuel})

    # hotel: the crew sleeps in a hotel every night, from a show until the next show (last stop: 1 night)
    for j, s in enumerate(shows):
        next_show = shows[j + 1]['show_date'] if j + 1 < len(shows) else s['show_date'] + timedelta(days=1)
        s['nights'] = (next_show - s['show_date']).days
        s['lodging_cost'] = 0
        for n in range(s['nights']):
            weekday = (s['show_date'] + timedelta(days=n)).weekday()           # Mon=0 ... Fri=4, Sat=5
            price = inp['weekend'][s['stop']] if weekday in (4, 5) else inp['weekday'][s['stop']]
            s['lodging_cost'] += rooms * price
    return shows, skipped

def route_table(shows, stops):
    t = pd.DataFrame(shows)
    t['city'] = stops.city[t.stop].values
    t['venue'] = stops.venue_name[t.stop].values
    t['hotel'] = stops.hotel_name[t.stop].values
    t['total_cost'] = t.fuel_cost + t.lodging_cost
    return t[['city', 'venue', 'show_date', 'drive_miles', 'fuel_cost', 'nights', 'hotel', 'lodging_cost', 'total_cost']].round(0)

# the simple plan: visit the cities in ranking order (best city first)
simple_order = list(range(inp['n']))
simple_shows, simple_skipped = schedule_route(simple_order, inp)
print('cities skipped (no date left):', stops.city[simple_skipped].tolist())
route_table(simple_shows, stops)

cities skipped (no date left): ['Orlando']


,city,venue,show_date,drive_miles,fuel_cost,nights,hotel,lodging_cost,total_cost
0,Portland,Echo Music Hall,2027-03-13,0.0,0.0,49,Summit Motel,18844,18844.0
1,Chicago,Northside Bowl,2027-05-01,2109.0,1015.0,15,Crown Suites,7780,8795.0
2,Seattle,Riverside Auditorium,2027-05-16,2080.0,1045.0,14,Grand Motel,7864,8909.0
3,Boston,Echo Music Hall,2027-05-30,2982.0,1493.0,5,Capitol Suites,2940,4433.0
4,Austin,Harbor Pavilion Hall,2027-06-04,2033.0,828.0,7,Harbor Motel,2924,3752.0
5,Minneapolis,Velvet Music Hall,2027-06-11,1254.0,479.0,7,Union Inn,2964,3443.0
6,Los Angeles,Golden Amphitheater,2027-06-18,1826.0,915.0,1,Capitol Inn,656,1571.0


In [14]:
# step 7: try EVERY possible order and keep the cheapest one
def find_best_route(inp):
    best = None
    for order in itertools.permutations(range(inp['n'])):
        shows, skipped = schedule_route(order, inp)
        cost = sum(s['fuel_cost'] + s['lodging_cost'] for s in shows)
        score = (len(skipped), cost)             # visiting every city matters more than saving money
        if best is None or score < best['score']:
            best = {'score': score, 'order': order, 'shows': shows, 'skipped': skipped}
    return best

best_route = find_best_route(inp)               # takes about 10 seconds
route_table(best_route['shows'], stops)

,city,venue,show_date,drive_miles,fuel_cost,nights,hotel,lodging_cost,total_cost
0,Los Angeles,Golden Amphitheater,2027-05-01,0.0,0.0,7,Capitol Inn,4672,4672.0
1,Austin,Harbor Pavilion Hall,2027-05-08,1472.0,719.0,7,Harbor Motel,2924,3643.0
2,Boston,Echo Music Hall,2027-05-15,2033.0,828.0,6,Capitol Suites,3544,4372.0
3,Orlando,Granite Theater,2027-05-21,1338.0,581.0,7,Lakeside Lodge,2932,3513.0
4,Chicago,Northside Bowl,2027-05-28,1183.0,516.0,1,Crown Suites,540,1056.0
5,Minneapolis,Velvet Music Hall,2027-05-29,426.0,180.0,7,Union Inn,2964,3144.0
6,Portland,Echo Music Hall,2027-06-05,1711.0,776.0,1,Summit Motel,376,1152.0
7,Seattle,Riverside Auditorium,2027-06-06,172.0,92.0,1,Grand Motel,564,656.0


In [15]:
# step 8: how much did the search save compared to the simple order?
def summary(shows, skipped):
    return {'cities visited': len(shows),
            'cities skipped': len(skipped),
            'tour days': (shows[-1]['show_date'] - shows[0]['show_date']).days + 1,
            'miles driven': round(sum(s['drive_miles'] for s in shows)),
            'fuel': round(sum(s['fuel_cost'] for s in shows)),
            'lodging': round(sum(s['lodging_cost'] for s in shows))}

compare = pd.DataFrame({'simple order (best city first)': summary(simple_shows, simple_skipped),
                        'cheapest order': summary(best_route['shows'], best_route['skipped'])})
compare.loc['total cost'] = compare.loc['fuel'] + compare.loc['lodging']
compare

,simple order (best city first),cheapest order
cities visited,7,8
cities skipped,1,0
tour days,98,37
miles driven,12284,8334
fuel,5776,3692
lodging,43972,18516
total cost,49748,22208


In [16]:
# step 9: do the same for every artist (steps 1-7 in one function)
def plan_tour(artist_id):
    stops = add_cheapest_hotel(add_gas_price(get_stops(artist_id)))
    inp = prepare_inputs(stops)
    return stops, find_best_route(inp)

artists = pd.read_csv('data/artist.csv').set_index('artist_id')
plans = {a: plan_tour(a) for a in artists.index}     # takes about 30 seconds
{a: [stops.city[k] for k in best['order'] if k not in best['skipped']] for a, (stops, best) in plans.items()}

{'A001': ['Los Angeles',
  'Austin',
  'Boston',
  'Orlando',
  'Chicago',
  'Minneapolis',
  'Portland',
  'Seattle'],
 'A002': ['Chicago',
  'Nashville',
  'Seattle',
  'Portland',
  'Las Vegas',
  'Los Angeles',
  'Houston',
  'Austin'],
 'A003': ['Houston',
  'Miami',
  'Philadelphia',
  'Detroit',
  'Milwaukee',
  'Los Angeles']}

In [17]:
# step 10: save the plans for the app (app/index.html reads app/data.js)
def tour_to_dict(artist_id, stops, best):
    out, prev = [], None
    for s in best['shows']:
        row = stops.loc[s['stop']]
        out.append({
            'city': row.city, 'state': row.state, 'venue': row.venue_name, 'venue_type': row.venue_type,
            'venue_id': row.venue_id, 'city_id': row.city_id, 'locality': row.locality, 'avg_ticket_price_usd': int(row.avg_ticket_price_usd),
            'capacity': int(row.capacity), 'expected_demand': int(row.expected_demand),
            'lat': float(row.venue_lat), 'lon': float(row.venue_lon),
            'show_date': s['show_date'].isoformat(), 'nights': int(s['nights']),
            'drive_miles': round(float(s['drive_miles']), 1),
            'gas_price_from': float(stops.gas_price[prev if prev is not None else s['stop']]),
            'gas_price_to': float(row.gas_price),
            'hotels': hotels[hotels.city_id == row.city_id][['hotel_name', 'rating', 'price_weekday_usd', 'price_weekend_usd']].to_dict('records'),
        })
        prev = s['stop']
    return {'artist_id': artist_id, 'artist_name': artists.artist_name[artist_id], 'genre': artists.primary_genre[artist_id],
            'stops': out, 'skipped': stops.city[best['skipped']].tolist()}

app_data = {
    'defaults': {'crew_size': crew_size, 'mpg': mpg, 'min_rating': min_rating},
    'window': {'start': tour_start.isoformat(), 'end': tour_end.isoformat()},
    'artists': [tour_to_dict(a, stops, best) for a, (stops, best) in plans.items()],
    'gas': gas_prices.round(2).to_dict(),                                                      # price per gallon by state
    'hotels': {cid: g.drop(columns='city_id').to_dict('records') for cid, g in hotels.groupby('city_id')},   # hotels by city
}

os.makedirs('app', exist_ok=True)
with open('app/data.js', 'w') as f:
    f.write('window.TOUR_DATA = ' + json.dumps(app_data, default=lambda o: o.item() if hasattr(o, 'item') else str(o)) + ';')

[(a['artist_name'], len(a['stops'])) for a in app_data['artists']]

[('Nova Reyes', 8), ('The Copper Pines', 8), ('Kairo Blaze', 6)]

### See it in the app

Open `app/index.html` in a browser. It shows the route on a map and lets you change the crew size, miles per
gallon and minimum hotel rating, and re-prices the tour.